# 💎 **Best Model Selection for Diamond Pricing**

Welcome to this notebook where we aim to find the best machine learning model for predicting diamond prices using the famous `diamonds` dataset from Seaborn.

![Diamonds](https://www.bing.com/images/create/generate-the-image-of-diamond-price-prediction-mod/1-68618bd997144bb3a406de02a13786e0?id=dahmCY2uqqtxCb8AXk3MuQ%3d%3d&view=detailv2&idpp=genimg&thId=OIG3.lmpr3WE1dj5d6TjDvHRF&skey=O4GHoJ6XcbmpT9moo_-Mk_nP8lLVF8U48xYN-fNCfo4&FORM=GCRIDP)



# **About the Author** 
Hello! I’m Asadullah Shehbaz — an aspiring Data Scientist on a journey of discovery and growth in the world of data.
I'm passionately exploring the realms of data science, diving deep into analytical techniques, machine learning, and real-world problem-solving to sharpen my skills and contribute meaningfully to this dynamic field.

> "I believe that community is the cornerstone of growth and innovation — by learning together, we can go further, faster, and stronger".


| Name               | Email                                               | LinkedIn                                                  | GitHub                                           | Kaggle                                        |
|--------------------|-----------------------------------------------------|-----------------------------------------------------------|--------------------------------------------------|-----------------------------------------------|
| **Asadullah Shehbaz**      |asadullahcreative@gmail.com  | [![LinkedIn Badge](https://img.shields.io/badge/LinkedIn-%23000000.svg?style=for-the-badge&logo=LinkedIn&logoColor=white)](https://www.linkedin.com/in/asadullah-shehbaz-18172a2bb/)  | [![GitHub Badge](https://img.shields.io/badge/GitHub-%23000000.svg?style=for-the-badge&logo=GitHub&logoColor=white)](https://github.com/AsadullahShehbaz)  | [![Kaggle Badge](https://img.shields.io/badge/Kaggle-%23000000.svg?style=for-the-badge&logo=Kaggle&logoColor=white)](https://www.kaggle.com/asadullahcreative)  |

---

# 💎 **Objective**

> **Can we accurately predict the price of a diamond based on its physical and quality attributes?**  
This notebook evaluates several machine learning models to predict `price` using features like `carat`, `cut`, `color`, and `clarity`.

---

# 🧭 **What This Notebook Covers**

### 1. 📦 Load the Diamonds Dataset  
- Loads the built-in `diamonds` dataset from the `seaborn` library.

---

### 2. 🧹 Data Preprocessing  
- Separates features into:
  - 🧮 Numerical: `carat`, `depth`, `table`, `x`, `y`, `z`
  - 🔤 Categorical: `cut`, `color`, `clarity`
- Applies preprocessing via a **`ColumnTransformer`**:
  - Scales numerical features.
  - Label encodes categorical features.

---

### 3. 🧪 Model Pipeline Setup  
- Builds a machine learning pipeline using `Pipeline()` from scikit-learn.
- Uses `GridSearchCV` with 5-fold cross-validation for **hyperparameter tuning**.
- Evaluates the following regression models:



1. 🔹 `LinearRegression` – Basic linear model
2. 🔹 `Ridge` – Linear regression with L2 regularization
3. 🔹 `Lasso` – Linear regression with L1 regularization
4. 🔹 `SVR` – Support Vector Regressor with linear and RBF kernels
5. 🔹 `KNeighborsRegressor` – K-Nearest Neighbors
6. 🌲 `RandomForestRegressor` – Ensemble of decision trees
7. 🌟 `GradientBoostingRegressor` – Boosting method for regression
8. ⚡ `XGBRegressor` – eXtreme Gradient Boosting (XGBoost)
9. 🎯 `AdaBoostRegressor` – Adaptive Boosting model
10. 🚀 `HistGradientBoostingRegressor` – Fast, histogram-based boosting from scikit-learn



---

### 4. 📊 Model Evaluation  
- For each model, it prints:
  - **Best cross-validated score**
  - **Best hyperparameters**
- Compares models using metrics:
  - **R² Score**
  - **Root Mean Squared Error (RMSE)**
  - **Mean Absolute Error (MAE)**

---

### 5. 🏆 Best Model Selection  
- Identifies the **best-performing model** based on validation scores.
- Retrains the final model on the full training set using optimal hyperparameters.
- Saves model for deployment.

---

## 💡 Key Learning Outcomes

- Learn how to build and tune **multiple ML regression models** using `Pipeline` and `GridSearchCV`.
- Understand which features and models work best for structured tabular data.
- Gain experience with model evaluation and performance comparison.
- Use ensemble models like **XGBoost** and **HistGradientBoosting** effectively.

---

# Import Libraries 

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error,mean_absolute_percentage_error

from sklearn.linear_model import LinearRegression, Ridge , Lasso
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from xgboost import XGBRegressor

import warnings
warnings.filterwarnings('ignore')


### 📥 Load and Preview Dataset


In [ ]:
df = sns.load_dataset("diamonds")
df = df.sample(n=1000, random_state=42)
df.head()


In [ ]:
# check the shape of the dataset
df.shape

### 🔍 Exploratory Data Analysis (EDA)

Let's understand the distribution of diamond prices and how features like `cut`, `color`, and `clarity` impact them.


In [ ]:
# Display dataset information to understand its structure
df.info()

Interpretations : 
1. There are 3 Categorical and 7 Numerical Features in the data
2. Total Rows are 53940
3. Total Columns are 10
4. There are no null values in the data
5. Categorical Columns are `cut`,`color`,`clarity`

In [ ]:
# Display summary statistics to understand the data distribution
df.describe() 

In [ ]:
categorical_cols = df.select_dtypes(include='category').columns.tolist()
numerical_cols = df.select_dtypes(include='number').columns.tolist()

print("Categorical:", categorical_cols)
print("Numerical:", numerical_cols)


### 📊 Univariate Analysis

In [ ]:
# Make Count Plot for Categorical Features to visualize their distributions
for col in categorical_cols:
    sns.countplot(x=col, data=df)
    plt.title(f'Count Plot for {col}')
    plt.xticks(rotation=45)
    plt.show()


In [ ]:
# Make histplot for numerical features to visualize their distributions
for col in numerical_cols:
    sns.histplot(df[col], kde=True)
    plt.title(f'Distribution of {col}')
    plt.show()


### 🧮Boxplots for Outliers Detection


In [ ]:
for col in numerical_cols:
    sns.boxplot(x=df[col])
    plt.title(f'Boxplot for {col}')
    plt.show()


### 🔥 Outlier Removal Using IQR

In [ ]:
# Make a copy of the original dataset
df_cleaned = df.copy()

# Define numerical columns
numerical_cols = df_cleaned.select_dtypes(include='number').columns.tolist()

# IQR method for outlier removal
for col in numerical_cols:
    Q1 = df_cleaned[col].quantile(0.25)
    Q3 = df_cleaned[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Filter out the outliers
    df_cleaned = df_cleaned[(df_cleaned[col] >= lower_bound) & (df_cleaned[col] <= upper_bound)]

# Final shape after outlier removal
print(f"Original Shape: {df.shape}")
print(f"Shape after Outlier Removal: {df_cleaned.shape}")


In [ ]:
df = df_cleaned

### 🧪 Bivariate Analysis

In [ ]:
# Make boxplots for numerical features
for col in categorical_cols:
    sns.boxplot(x=col, y='price', data=df)
    plt.title(f'Price vs {col}')
    plt.xticks(rotation=45)
    plt.show()


In [ ]:
# Make pairplot to visualize relationships between numerical features
sns.pairplot(df[numerical_cols + ['cut']], hue='cut')
plt.suptitle('Pairplot of Numerical Features with Cut', y=1.02)
plt.show()


In [ ]:
# Correlation Matrix & Heatmap
corr_matrix = df[numerical_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix')
plt.show()

In [ ]:
# 📐 Price Relationship with Dimensions
sns.scatterplot(x='carat', y='price', data=df, hue='cut')
plt.title('Carat and Price')
plt.show()

sns.scatterplot(x='x', y='price', data=df)
plt.title('Length (x) and Price')
plt.show()


In [ ]:
# make barplot of price and cut columns
plt.Figure(figsize=(10,6))
sns.barplot(x='cut', y='price', data=df)
plt.title('Price of Diamonds by Cut')
plt.xlabel('Cut')
plt.ylabel('Price')
plt.show()

In [ ]:
# 🧼 Check for Duplicates & Zero Values
print("Duplicate Rows: ", df.duplicated().sum())
print("-" * 20)
# Check zero or unrealistic values
for col in ['x', 'y', 'z']:
    print(f"{col} zero count: ", (df[col] == 0).sum())


### 🧮 Feature Engineering


In [ ]:
# Create a new feature 'volume' as the product of dimensions x, y, and z 
# df['volume'] = df['x'] * df['y'] * df['z']
# df['volume'].head()

In [ ]:
X = df.drop('price', axis=1)  # Features (excluding price column)
y = df['price']  # Target variable ( price column)

# Define column types for preprocessing
categorical_cols = X.select_dtypes(include='object').columns.tolist()
numerical_cols = X.select_dtypes(include='number').columns.tolist()


### ⚙️ Data Preprocessing with Pipelines


In [ ]:
# Preprocessing pipelines for numerical and categorical features
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine preprocessing steps for numerical and categorical features
preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_transformer, numerical_cols),
    ('cat', categorical_transformer, categorical_cols)
])


### 🤖 Define Models and Hyperparameter Grids


In [ ]:
#  Define models and hyperparameter grids

models = {
    "LinearRegression": {
        "model": LinearRegression(),
        "params": {}  # No hyperparams for basic linear model
    },
    "RandomForest": {
        "model": RandomForestRegressor(random_state=42),
        "params": {
            "model__n_estimators": [50, 100],
            "model__max_depth": [10, 20],
        }
    },
    "XGBoost": {
        "model": XGBRegressor(random_state=42, objective='reg:squarederror'),
        "params": {
            "model__n_estimators": [50, 100],
            "model__max_depth": [3, 6],
            "model__learning_rate": [0.1, 0.3]
        }
    },
    "Ridge": {
        "model": Ridge(),
        "params": {
            "model__alpha": [0.1, 1.0, 10.0]
        }
    },
    "Lasso": {
        "model": Lasso(),
        "params": {
            "model__alpha": [0.01, 0.1, 1.0]
        }
    },
    "SVR": {
        "model": SVR(),
        "params": {
            "model__kernel": ['linear', 'rbf'],
            "model__C": [0.1, 1, 10],
            "model__gamma": ['scale', 'auto']
        }
    },
    "KNeighbors": {
        "model": KNeighborsRegressor(),
        "params": {
            "model__n_neighbors": [3, 5, 7],
            "model__weights": ['uniform', 'distance']
        }
    },
    "GradientBoosting": {
        "model": GradientBoostingRegressor(random_state=42),
        "params": {
            "model__n_estimators": [100, 200],
            "model__learning_rate": [0.05, 0.1],
            "model__max_depth": [3, 5]
        }
    },
    "AdaBoost": {
        "model": AdaBoostRegressor(random_state=42),
        "params": {
            "model__n_estimators": [50, 100],
            "model__learning_rate": [0.5, 1.0]
        }
    },
    "HistGradientBoosting": {
        "model": HistGradientBoostingRegressor(random_state=42),
        "params": {
            "model__learning_rate": [0.05, 0.1],
            "model__max_iter": [100, 200],
            "model__max_depth": [None, 10]
        }
    }
}


### 🔄 Model Training with Cross-Validation


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:

# Store results (model name and performance metrics)
results = []

for name, model_params in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model_params['model'])
    ])

    # Hyperparameter tuning
    grid_search = GridSearchCV(pipeline, model_params['params'], cv=5, scoring='r2',error_score='raise', n_jobs=-2)
    grid_search.fit(X_train, y_train)

    # Evaluate on test set
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)

    results.append({
        "Model": name,
        "Best Params": grid_search.best_params_,
        "R2": r2_score(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "MAE": mean_absolute_error(y_test, y_pred),
        "MAPE": mean_absolute_percentage_error(y_test, y_pred)
    })

In [ ]:
# Final results
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='R2', ascending=False)
results_df

### 📊 Model Comparison Visualization


In [ ]:
# Visualize the results using a bar plot
sns.barplot(results_df,y='Model',x='R2')
plt.title('Model Performance Comparison')
plt.xlabel('R2 Score')
plt.ylabel('Model')
plt.show()

### ✅ **Interpretation**

- The Best Model is XGBoost with highest r2 score (0.87).
- In this analysis, ensemble models like **XGBoost** and **AdaBoost** outperformed simple linear models.
- This pipeline-based approach ensures reproducibility and scalability for future datasets.
- Further improvements may include feature engineering, outlier removal, or advanced tuning with more data.


# Retrain the best model 

In [ ]:
# retrain the best model on the full training set
best_model_name = results_df.iloc[0]['Model']
best_params = results_df.iloc[0]['Best Params']

# Get the model and its parameter grid
model_info = models[best_model_name]
model_instance = model_info['model']

# Build the pipeline
pipeline = Pipeline([
	('preprocessor', preprocessor),
	('model', model_instance)
])

# Set the best hyperparameters (if any)
if best_params:
	pipeline.set_params(**best_params)

# Fit on the full training set
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

# print the performance metrics of the best model 
print(f"Best Model: {best_model_name}")
print(f"R2 Score: {r2_score(y_test, y_pred)}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred))}")
print(f"MAE: {mean_absolute_error(y_test, y_pred)}")
print(f"MAPE: {mean_absolute_percentage_error(y_test, y_pred)}")

# Save the Model 

In [ ]:
import pickle 
# Save the best model to a file
best_model = results_df.iloc[0]['Model']
pickle.dump(best_model, open('best_model.pkl', 'wb'))

---

# ✅ **Conclusion & Final Thoughts**

In this notebook, we:

- 📊 Explored the **Diamonds dataset** and performed thorough **EDA**
- 🧹 Applied preprocessing pipelines using `ColumnTransformer` for both numerical and categorical features
- 🔧 Evaluated a wide range of **regression models** using `GridSearchCV` for **hyperparameter tuning**
- 🏆 Identified the best-performing model based on validation metrics such as **R² Score**, **RMSE**, and **MAE**

---

### 🚀 Key Outcomes:

- **Modeling pipelines** made the process modular, clean, and scalable.
- **Ensemble models** (like Random Forest, XGBoost, and HistGradientBoosting) showed strong predictive performance.
---

> 🧠 *This project demonstrates how to build an end-to-end regression pipeline using real-world structured data — from raw input to tuned, validated model predictions.*  

---
